# Practical P17: Install LangChain & Build First Pipeline (Chains & LCEL)
**Course**: PGDCA — Hands-On Large Language Models  
**Unit 3**: LLM Frameworks for Application Development  
**Syllabus Topic**: 3.1 LangChain framework: chains, memory, output parsers  
**Model Provider**: Google Gemini (`gemini-1.5-flash`) via `langchain-google-genai`  
**Learning Outcome**: Master the LangChain framework ecosystem, configure real Google Gemini API models, master the LangChain Expression Language (LCEL) syntax, and build production-ready `Prompt -> Model -> Output Parser` pipelines.

## Part 1: Theoretical Foundations — The LangChain Framework

### 1.1 Why Do We Need an Orchestration Framework?
In Units 1 and 2, we invoked LLMs using raw HTTP requests. While effective for simple prompts, real-world production AI applications face major challenges:
* **Vendor Lock-in**: Switching model providers requires rewriting payloads, parsing routines, and exception handlers.
* **Complex Multi-Step Logic**: Real applications require piping outputs of one prompt as inputs to another, retrieving external documents, and performing validations.
* **Fragile Outputs**: LLMs produce free-form strings. Software systems, databases, and web APIs require predictable structures.

**LangChain** solves these problems by providing standardized, modular interfaces for every component of an LLM application.

### 1.2 The Modern LangChain Ecosystem
Modern LangChain is split into lightweight, focused packages:
1. **`langchain-core`**: Base abstractions (`Runnable`, `PromptTemplate`, `BaseChatModel`, `BaseOutputParser`), minimal dependencies.
2. **`langchain`**: Pre-built chains, cognitive workflows, and memory managers.
3. **`langchain-community`**: Third-party integrations for vector stores, tools, and retrievers.
4. **Partner Packages (`langchain-google-genai`, `langchain-openai`)**: Direct vendor-maintained adapters for maximum performance.

```mermaid
graph LR
    Input[User Input] --> Prompt[PromptTemplate]
    Prompt --> Model[Google Gemini ChatModel]
    Model --> Parser[OutputParser]
    Parser --> Output[Clean String / JSON Result]
```

In [1]:
# Part 1 Code: Package verification and API Key inspection
import os
import langchain
import langchain_core
import langchain_google_genai
from dotenv import load_dotenv

# Load environment variables from .env file in workspace root
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

print(f"✅ LangChain Version: {langchain.__version__}")
print(f"✅ LangChain Core Version: {langchain_core.__version__}")
print(f"✅ LangChain Google GenAI Version: {langchain_google_genai.__version__}")

if api_key and api_key.strip():
    print(f"🔑 GEMINI_API_KEY Status: Configured ({api_key[:6]}...{api_key[-4:]})")
else:
    print("⚠️ GEMINI_API_KEY Status: Not set yet in .env file.")
    print("👉 To run live Gemini calls: open '.env' and add: GEMINI_API_KEY=your_key_here")

✅ LangChain Version: 1.4.0
✅ LangChain Core Version: 1.6.3
✅ LangChain Google GenAI Version: 4.4.0
🔑 GEMINI_API_KEY Status: Configured (AQ.Ab8...ZLIg)


## Part 2: Core Components of an LCEL Pipeline

A canonical LangChain pipeline consists of three core components connected sequentially:

1. **Prompt Template (`ChatPromptTemplate`)**:
   Transforms raw user variables into formatted messages (`SystemMessage`, `HumanMessage`). It separates prompt engineering from application code.
2. **Chat Model (`ChatGoogleGenerativeAI`)**:
   The reasoning engine using Google's state-of-the-art `gemini-1.5-flash` model.
3. **Output Parser (`StrOutputParser`)**:
   Extracts the string response from the model's `AIMessage` object and strips superfluous metadata.

### The LCEL Pipe (`|`) Syntax
LangChain Expression Language (LCEL) composes components using the Unix pipe operator (`|`). Under the hood, each component implements the **`Runnable` interface**:

$$\text{chain} = \text{prompt} \mid \text{model} \mid \text{parser}$$

When you invoke `chain.invoke(input)`, LCEL automatically:
1. Passes `input` to `prompt` $\rightarrow$ returns a `PromptValue`.
2. Passes `PromptValue` to `model` (Google Gemini) $\rightarrow$ returns an `AIMessage`.
3. Passes `AIMessage` to `parser` $\rightarrow$ returns a clean string.

In [4]:
# Part 2 Code: Building a Real Google Gemini LCEL Pipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Define Prompt Template with System and Human roles
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a knowledgeable computer science professor teaching PGDCA students. Answer concisely."),
    ("human", "Explain the concept of '{topic}' in 2 sentences.")
])

# 2. Initialize Real Google Gemini Chat Model
# Uses key from .env if present, otherwise uses placeholder until key is added
chat_model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    api_key=api_key or "AIzaSy_placeholder_until_env_key_set",
    temperature=0.7
)

# 3. Initialize Output Parser
output_parser = StrOutputParser()

# 4. Compose LCEL Chain using the pipe operator (|)
first_chain = prompt_template | chat_model | output_parser

# 5. Invoke the pipeline with user parameters
try:
    response = first_chain.invoke({"topic": "LangChain Expression Language (LCEL)"})
    print("=" * 60)
    print("🎯 REAL GOOGLE GEMINI LCEL RESULT:")
    print("=" * 60)
    print(response)
except Exception as e:
    print("=" * 60)
    print("ℹ️ GEMINI API EXECUTION STATUS:")
    print("=" * 60)
    print("To execute this live against Google Gemini, please add your GEMINI_API_KEY to the .env file.")
    print(f"Notice: {e}")

/Users/galaxyofai/Desktop/Github/Hands-On-LLM-PGDCA/venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


ℹ️ GEMINI API EXECUTION STATUS:
To execute this live against Google Gemini, please add your GEMINI_API_KEY to the .env file.
Notice: Error calling model 'gemini-3.5-flash-lite' (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED'}}


## Part 3: Multi-Step Sequential Chains with `RunnablePassthrough`

Real applications often require multiple reasoning steps:
1. **Step 1**: Prompt Gemini to generate a technical definition of a concept.
2. **Step 2**: Take that definition and prompt Gemini to generate 2 practical lab assignments.

LCEL handles branching and data routing seamlessly using Python dictionaries and `RunnablePassthrough`.

In [3]:
# Part 3 Code: Multi-Step Pipeline with Google Gemini
from langchain_core.runnables import RunnablePassthrough

# Prompt 1: Generate technical definition
summary_prompt = ChatPromptTemplate.from_template(
    "Provide a 1-sentence technical definition of {concept}."
)

# Prompt 2: Generate practical lab exercises based on the definition
exercises_prompt = ChatPromptTemplate.from_messages([
    ("system", "You design university computer science lab assignments."),
    ("human", "Concept: {concept}\nDefinition: {definition}\nList 2 practical lab assignments for PGDCA students:")
])

# Define Step 1 Chain
summary_chain = summary_prompt | chat_model | StrOutputParser()

# Compose Multi-Step Sequential Chain
# RunnablePassthrough() retains the original concept while summary_chain computes 'definition'
full_pipeline = (
    {"concept": RunnablePassthrough(), "definition": summary_chain}
    | exercises_prompt
    | chat_model
    | StrOutputParser()
)

try:
    result = full_pipeline.invoke("Retrieval-Augmented Generation (RAG)")
    print("=" * 60)
    print("🚀 REAL GEMINI MULTI-STEP PIPELINE OUTPUT:")
    print("=" * 60)
    print(result)
except Exception as e:
    print("ℹ️ Set your GEMINI_API_KEY in .env to run this multi-step chain live with Gemini.")
    print(f"Notice: {e}")

ℹ️ Set your GEMINI_API_KEY in .env to run this multi-step chain live with Gemini.
Notice: Error calling model 'gemini-1.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}


## Part 4: Hands-On Student Exercise

**Objective**: Build a two-stage "Code Drafter & Auditor" pipeline using Google Gemini:
1. **Stage 1 (Code Drafter)**: Prompt Gemini to generate a clean Python function for a given task.
2. **Stage 2 (Code Auditor)**: Pass the generated Python code to an auditor prompt that evaluates time complexity and verifies edge case robustness.

In [4]:
# Student Exercise: Code Drafter & Auditor Pipeline with Gemini
drafter_prompt = ChatPromptTemplate.from_template(
    "Write a clean Python function for: {task}. Return only code with a short docstring."
)

auditor_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a senior code reviewer. Analyze the code for time/space complexity and edge cases in 2-3 bullet points."),
    ("human", "Task: {task}\nCode:\n{code}\nProvide code review verdict:")
])

# Stage 1: Drafter chain
drafter_chain = drafter_prompt | chat_model | StrOutputParser()

# Stage 2: Combined Pipeline
code_review_pipeline = (
    {"task": RunnablePassthrough(), "code": drafter_chain}
    | auditor_prompt
    | chat_model
    | StrOutputParser()
)

try:
    review_output = code_review_pipeline.invoke("Check if a string is a palindrome")
    print("=" * 60)
    print("📋 REAL GEMINI CODE REVIEW PIPELINE OUTPUT:")
    print("=" * 60)
    print(review_output)
except Exception as e:
    print("ℹ️ Set your GEMINI_API_KEY in .env to run this exercise live with Gemini.")
    print(f"Notice: {e}")

ℹ️ Set your GEMINI_API_KEY in .env to run this exercise live with Gemini.
Notice: Error calling model 'gemini-1.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}
